In [ ]:
from datasets import load_dataset

In [ ]:
data = load_dataset("cornell-movie-review-data/rotten_tomatoes") # 注意这里要用新的名称
data

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    device_map="auto"
)

In [ ]:
prompt = "Is the following sentence negative or positive?"
data = data.map(lambda example: {"t5": prompt + example["text"]}) # 给data数据集新加一个字段
data

In [ ]:
import numpy as np
from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset

y_pred = []
for text in tqdm(KeyDataset(data["test"], "t5"), total=len(data["test"])):
  tokens = tokenizer(
      text,
      return_tensors="pt",
      truncation=True,
  ).to(model.device)
  output = model.generate(
      **tokens,
      max_new_tokens=10,
  )
  result = tokenizer.decode(
      output[0],
      skip_special_tokens=True
  )
  y_pred.append(0 if result == "negative" else 1)

In [ ]:
from sklearn.metrics import classification_report

def evaluate_performance(y_true, y_pred):
  performance = classification_report(
      y_true, y_pred,
      target_names=["Negative Review", "Positive Review"],
  )
  print(performance)

In [ ]:
evaluate_performance(data["test"]["label"], y_pred)

In [ ]:
import openai

client = openai.OpenAI(api_key="")

In [ ]:
def chatgpt_generation(prompt, document, model="gpt-5.4"):
  messages = [
      {
          "role": "system",
          "content": "You are a helpful assistant",
      },
      {
          "role": "user",
          "content": prompt.replace("[DOCUMENT]", document)
      }
  ]

  chat_completion = client.chat.completions.create(
      messages = messages,
      model=model,
      temperature=0,
  )
  return chat_completion.choices[0].message.content

In [33]:
prompt = """
Predict whether the following document is a postive or negative movie review:

[DOCUMENT]

If it is postive return 1 and if it is negative return 0. Do not give other answers.
"""

predictions = [
    chatgpt_generation(prompt, doc) for doc in tqdm(data["test"]["text"])
]

In [ ]:
y_pred = [int(pred) for pred in predictions]

evaluate_performance(data["test"]["label"], y_pred)